In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from google.colab import drive

drive.mount("/content/drive")

#spilloverを考慮したSCM感度分析
"""
この感度分析の目的は、「ドナー（比較対象）介入の波及効果があるときに、SCMの推定精度がどれほど悪化するか」を確認すること。
SCMの計算結果は「処置群の実績値」と「合成コントロール（反実仮想）の予測値」の差分（Gap）の平均値。
そのため商圏が重複する対照群Bへの流出によって処置群Aの数値が減っているかどうかに関わらず、
「ドナー側の来訪者が増えてしまうこと自体が分析の敵（処置効果の過小評価につながってしまう）」であるため、
まずは介入によるドナー側の増加（汚染）のみをシンプルにシミュレーションする
"""
input_path = Path("/content/drive/MyDrive/因果推論/h3_mesh_panel.csv")
output_dir = input_path.parent

intervention_date = pd.Timestamp("2025-01-01")
#介入が未処置の領域にどの程度波及して影響を与えるかをシミュレーションするための感度分析において使用される変数
spillover_strengths = [0.00, 0.25, 0.50, 1.00]
#「スピルオーバーの影響を受けている可能性が高いドナー（対照群）を、分析から除外するためのしきい値」の
exclusion_thresholds = [1.01, 0.70, 0.60, 0.50, 0.30]

#このCSVには実測の距離・隣接関係・商圏重複率がないため、教材用に
#donor_groupとJaccard係数を明示的に設定する。駅名や実地域は表さない。
teaching_groups = {
    "donor_group_1": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(7, 13)], "overlap_score": 0.75},
    "donor_group_2": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(13, 19)], "overlap_score": 0.45},
    "donor_group_3": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(19, 25)], "overlap_score": 0.25},
    "donor_group_4": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(25, 31)], "overlap_score": 0.20},
    "donor_group_5": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(31, 37)], "overlap_score": 0.65},
}


def rmspe(values):
    values = np.asarray(values, dtype=float)
    return float(np.sqrt(np.mean(values ** 2)))


def fit_scm(panel, strength, threshold, true_effect):
    work = panel.copy()
    work["assumed_spillover"] = np.where(
        (work["treated"] == 0) & (work["date"] >= intervention_date),
        #商圏重複率 * 未処置の領域に波及して影響する効果 * 真の処置効果
        work["overlap_score"] * strength * true_effect,0.0)

    #シミュレーション用に人工的に作り出した、汚染（波及）後の来訪者数
    """
    もし周辺エリアがこれだけ汚染されていたら、SCMはどれくらい間違った答え（推定誤差）を出してしまうのか？」を検証するために、
    あえて汚染されたこの列を分析の入力データとして使用します。
    """
    work["visitors_scenario"] = work["visitors"] + work["assumed_spillover"]

